In [34]:
import pandas as pd
import numpy as np
import re

Empezamos con el dataset de METEOROLOGIA

In [ ]:
meteo_limpio = pd.read_csv(r"C:\Users\bario\OneDrive\Documentos\UC3M\PROYECTO EN CIENCIA DE DATOS\proyecto_emergencias-20260915T155207Z-1-001\proyecto_emergencias\meteorologia.csv") 
meteo_limpio.head()
#todas las modificaciones aqui
meteo_limpio = meteo_limpio.copy()

Primero hacer un analisis general del dataset y sus solumnas.
1) Evaluar la estructura global mediante una matriz integrada de tipos, nulos y control de duplicados. 
2) Aplicar filtros lógicos para detectar inconsistencias físicas (temp max no puede ser menor que la min, humedades no pueden estar fuera del rango 0-100, vientos/lluvias negativos). 
3) Extraer el rango de valores absolutos (mínimos y máximos) para identificar outliers. 
4) Analisis del sintaxis de la columna fecha (separadores y fallos de conversión NaT) 
5) Analisis de la columna condicion (caja de texto, tildes y variedad de categorías).

In [95]:
#ESTRUCTURA GENERAL, NULOS Y DUPLICADOS
print("===== 1. ANALISIS GENERAL, NULL, DUPLICADOS =====")
resumen_estructura = pd.DataFrame({
    'Tipo': meteo_limpio.dtypes,
    'No-Nulos': meteo_limpio.notnull().sum(),
    'Nulos': meteo_limpio.isnull().sum(),
})
print(resumen_estructura)
print(f"\nFilas duplicadas exactas: {meteo_limpio.duplicated().sum()}")


#ANALISIS DE DATOS CON COHERENCIA FÍSICA Y RANGOS
print("\n===== 2. COHERENCIA FÍSICA Y RANGOS =====")
temp_raras = meteo_limpio[meteo_limpio['temperatura_min'] > meteo_limpio['temperatura_max']]
humedad_fuera = meteo_limpio[(meteo_limpio['humedad_relativa'] < 0) | (meteo_limpio['humedad_relativa'] > 100)]
precip_neg = meteo_limpio[meteo_limpio['precipitacion_mm'] < 0]
viento_neg = meteo_limpio[meteo_limpio['viento_max_kmh'] < 0]

print(f"- Días con Temp. Mínima MAYOR que la Máxima: {len(temp_raras)}")
print(f"- Días con Humedad fuera de rango (0-100%): {len(humedad_fuera)}")
print(f"- Días con Precipitaciones negativas: {len(precip_neg)}")
print(f"- Días con Viento negativo: {len(viento_neg)}")


#MÍNIMOS Y MÁXIMO
print("\n===== 3. VALORES MÍNIMOS Y MÁXIMOS =====")
columnas_numericas = ['temperatura_max', 'temperatura_min', 'temperatura_media', 
                       'precipitacion_mm', 'humedad_relativa', 'viento_max_kmh', 
                       'presion_hpa', 'indice_uv']
print(meteo_limpio[columnas_numericas].describe().T[['min', 'max']])


#ANALISIS COLUMNA FECHA
print("\n===== 4. COLUMNA FECHAS =====")
fechas_str = meteo_limpio['fecha'].astype(str)
con_barra = fechas_str.str.contains('/').sum()
con_guion = fechas_str.str.contains('-').sum()
patron_dd_mm_yyyy = r'^\d{2}-\d{2}-\d{4}$'
no_cumplen_formato = (~fechas_str.str.match(patron_dd_mm_yyyy)).sum()
print(f"- Fechas separadas por '/': {con_barra}")
print(f"- Fechas separadas por '-': {con_guion}")
print(f"- Fechas que no cumplen formato DD MM YYYY: {no_cumplen_formato}")


#ANALISIS COLUMNA CONDICIÓN
print("\n===== 5. COLUMNA CONDICIÓN =====")
condiciones_raw = meteo_limpio['condicion'].dropna().tolist()

con_tilde = sum(1 for x in condiciones_raw if re.search(r'[áéíóúÁÉÍÓÚ]', str(x)))
con_espacios = sum(1 for x in condiciones_raw if str(x) != str(x).strip())
con_minusculas = sum(1 for x in condiciones_raw if not str(x).isupper())

print(f"Valores ÚNICOS detectados (sin limpiar):")
print(meteo_limpio['condicion'].unique())

print(f"\nInconsistencias encontradas:")
print(f"- Registros con minúsculas/mezcla de caja: {con_minusculas}")
print(f"- Registros con tildes: {con_tilde}")
print(f"- Registros con espacios sobrantes: {con_espacios}")

===== 1. ANALISIS GENERAL, NULL, DUPLICADOS =====
                      Tipo  No-Nulos  Nulos
fecha                  str       366      0
temperatura_max    float64       358      8
temperatura_min    float64       362      4
temperatura_media  float64       366      0
precipitacion_mm   float64       349     17
humedad_relativa   float64       353     13
viento_max_kmh     float64       359      7
presion_hpa        float64       366      0
indice_uv            int64       366      0
condicion              str       366      0

Filas duplicadas exactas: 0

===== 2. COHERENCIA FÍSICA Y RANGOS =====
- Días con Temp. Mínima MAYOR que la Máxima: 0
- Días con Humedad fuera de rango (0-100%): 0
- Días con Precipitaciones negativas: 0
- Días con Viento negativo: 0

===== 3. VALORES MÍNIMOS Y MÁXIMOS =====
                     min     max
temperatura_max      4.3    44.5
temperatura_min     -6.2    40.9
temperatura_media   -0.9    41.1
precipitacion_mm     0.0    17.1
humedad_relativa    20.0

OBSERVACIONES
1. Fechas Corruptas: Hay 246 fechas que no cumplen el formato adecuado debido la inversión de días/meses/años. Solucion --> homogeneizar la conversión para no perder el 67% de los datos. Asimismo hay 40 separadas poor /, que habra que pasar a - para tener os datos homogeneos.
2. Incoherencias Físicas y Nulos: Se detectan nulos en precipitación (9), humedad (8) y viento (7). Además, se observan 10 días con minimos mayores que maximos. Tmabien máximas de 59.9, mínimas de -22.2, humedades del 146 y precipitaciones negativas, lo cual no es coherente. Solución --> corrección/interpolación.
3. Fragmentación Categórica: El campo condicion presenta 18 valores distintos por mezcla de idiomas, sinónimos redundantes y falta de formato (358 de 366 registros con minúsculas/mezcla). Solución --> Elegir un formato (idioma, may/min) y hacer transformaciones de texto.

**SOLUCIÓN FORMATO FECHA**


In [84]:
#Función flexible para detectar el origen de la fecha
def parsear_fecha_flexible(fecha_str):
    if pd.isnull(fecha_str):
        return pd.NaT
    fecha_str = str(fecha_str).strip()
    
    # Si la cadena empieza por año 
    if fecha_str.startswith('2024'):
        return pd.to_datetime(fecha_str, format='mixed', dayfirst=False)
    # Si empieza por día 
    else:
        return pd.to_datetime(fecha_str, format='mixed', dayfirst=True)

# Aplicar el parseo flexible
meteo_limpio['fecha_clean'] = meteo_limpio['fecha'].apply(parsear_fecha_flexible)

# Formatear la fecha al estándar exacto DD-MM-YYYY
meteo_limpio['fecha'] = meteo_limpio['fecha_clean'].dt.strftime('%d-%m-%Y')

# Ordenar cronológicamente y descartar la auxiliar
meteo_limpio = meteo_limpio.sort_values('fecha_clean').reset_index(drop=True)
meteo_limpio = meteo_limpio.drop(columns=['fecha_clean'])

# VERIFICACIÓN
print("===== RESULTADOS TRAS EDITAR FECHAS =====")
print(f"- Fechas mal formateadas restantes: {meteo_limpio['fecha'].isnull().sum()}")
print(f"- Separadores '/' restantes: {meteo_limpio['fecha'].str.contains('/').sum()}")
print(f"- Fechas con formato con guion '-': {meteo_limpio['fecha'].str.contains('-').sum()}")
print(f"- Fechas/Días duplicados en el dataset: {meteo_limpio['fecha'].duplicated().sum()}")

# Validar la secuencia de los 366 días de 2024
fechas_dt = pd.to_datetime(meteo_limpio['fecha'], format='%d-%m-%Y')
rango_esperado = pd.date_range(start='2024-01-01', end='2024-12-31', freq='D')
dias_faltantes = len(rango_esperado) - len(fechas_dt.unique())

print(f"- Días faltantes en el año 2024 (esperados 366): {dias_faltantes}")


===== RESULTADOS TRAS EDITAR FECHAS =====
- Fechas mal formateadas restantes: 0
- Separadores '/' restantes: 0
- Fechas con formato con guion '-': 366
- Fechas/Días duplicados en el dataset: 0
- Días faltantes en el año 2024 (esperados 366): 0


Solución realizada en Fechas:
- **Transformación:** Implementación de una función condicional flexible que analiza la orientación del registro (`YYYY...` vs. `DD...`) para estandarizar la columna al formato único `DD-MM-YYYY`. Además de transformar los / en -.
- **Resultado:** Formato homogeno en todas las fechas del año. Cubriendo una totalidad de los 366 días de 2024, sin duplicados ni Null.

**SOLUCIÓN SEMÁNTICA CONDICIÓN**

In [85]:
#Definir sinonimos
mapa_condicion = {
    # Despejado
    'DESPEJADO': 'DESPEJADO', 'DESPEJADO': 'DESPEJADO', 'SOLEADO': 'DESPEJADO', 'CLEAR': 'DESPEJADO',
    # Nublado
    'NUBLADO': 'NUBLADO', 'PARCIALMENTE NUBLADO': 'NUBLADO', 'CUBIERTO': 'NUBLADO', 'NIEBLA': 'NUBLADO', 'CLOUDY': 'NUBLADO',
    # Lluvia
    'LLUVIA': 'LLUVIA', 'LLUVIOSO': 'LLUVIA', 'PRECIPITACIÓN': 'LLUVIA', 'PRECIPITACION': 'LLUVIA', 'RAIN': 'LLUVIA',
    # Tormenta
    'TORMENTA': 'TORMENTA'
}

#Normalizar texto: eliminar espacios, tildes y convertir a mayúsculas
condicion_prep = (
    meteo_limpio['condicion']
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace('Á', 'A').str.replace('É', 'E').str.replace('Í', 'I').str.replace('Ó', 'O').str.replace('Ú', 'U')
)

#Aplicar cambios
meteo_limpio['condicion'] = condicion_prep.map(mapa_condicion)

# VERIFICACIÓN
print("===== RESULTADOS TRAS EDITAR 'CONDICION' =====")
print(f"- Nulos en 'condicion': {meteo_limpio['condicion'].isnull().sum()}")
print(f"- Categorías únicas finales ({len(meteo_limpio['condicion'].unique())}):")
print(meteo_limpio['condicion'].value_counts())

===== RESULTADOS TRAS EDITAR 'CONDICION' =====
- Nulos en 'condicion': 0
- Categorías únicas finales (4):
condicion
DESPEJADO    162
NUBLADO      156
LLUVIA        40
TORMENTA       8
Name: count, dtype: int64


Solución realizada en 'condicion':
- **Estandarización y Mapeo:** Normalización del texto (eliminación de tildes, espacios sobrantes y conversión a mayúsculas) y consolidación de 18 variantes heterogéneas (sinónimos, mezcla de idiomas como inglés/español) en 4 categorías : `DESPEJADO`, `NUBLADO`, `LLUVIA` y `TORMENTA`.
- **Resultado:** 0 valores nulos generados y reducción exitosa del ruido categórico, obteniendo una distribución limpia y equilibrada para el modelado (162 Despejado, 156 Nublado, 40 Lluvia y 8 Tormenta).

**SOLUCIÓN INCOHERENCIAS TEMP, PRECIP, HUMEDAD, VIENTO**

In [86]:

# Seleccionamos las columnas relevantes para la inspección
cols = ['fecha', 'temperatura_max', 'temperatura_min', 'temperatura_media', 
        'precipitacion_mm', 'humedad_relativa', 'viento_max_kmh', 'condicion']

# 1. Temperatura Mínima MAYOR que la Máxima
print("===== 1. INCOHERENCIA: TEMP. MÍNIMA > TEMP. MÁXIMA =====")
df_min_gt_max = meteo_limpio[meteo_limpio['temperatura_min'] > meteo_limpio['temperatura_max']]
print(f"Total encontrados: {len(df_min_gt_max)}")
display(df_min_gt_max[cols])

# 2. Temperaturas extremas inverosímiles (> 50°C o < -20°C)
print("\n===== 2. VALORES EXTREMOS EN TEMPERATURA (>50°C o <-20°C) =====")
df_temp_loca = meteo_limpio[(meteo_limpio['temperatura_max'] > 50) | (meteo_limpio['temperatura_min'] < -20)]
print(f"Total encontrados: {len(df_temp_loca)}")
display(df_temp_loca[cols])

# 3. Precipitaciones Negativas (< 0 mm)
print("\n===== 3. PRECIPITACIONES NEGATIVAS =====")
df_precip_neg = meteo_limpio[meteo_limpio['precipitacion_mm'] < 0]
print(f"Total encontrados: {len(df_precip_neg)}")
display(df_precip_neg[cols])

# 4. Humedad Relativa fuera de Rango (< 0% o > 100%)
print("\n===== 4. HUMEDAD FUERA DE RANGO (0-100%) =====")
df_hum_rara = meteo_limpio[(meteo_limpio['humedad_relativa'] < 0) | (meteo_limpio['humedad_relativa'] > 100)]
print(f"Total encontrados: {len(df_hum_rara)}")
display(df_hum_rara[cols])

# 5. Valores Nulos o Faltantes (NaN)
print("\n===== 5. REGISTROS CON VALORES NULOS (NaN) =====")
df_nulos = meteo_limpio[meteo_limpio.isnull().any(axis=1)]
print(f"Total encontrados: {len(df_nulos)}")
display(df_nulos[cols])

===== 1. INCOHERENCIA: TEMP. MÍNIMA > TEMP. MÁXIMA =====
Total encontrados: 10


,fecha,temperatura_max,temperatura_min,temperatura_media,precipitacion_mm,humedad_relativa,viento_max_kmh,condicion
43,13-02-2024,11.8,16.576725,3.0,0.0,43.0,2.6,DESPEJADO
60,01-03-2024,14.2,16.205074,11.5,0.0,41.0,13.3,LLUVIA
122,02-05-2024,35.0,39.225850,30.0,0.0,31.0,15.7,DESPEJADO
132,12-05-2024,29.9,33.515356,26.2,0.0,84.0,1.2,NUBLADO
200,19-07-2024,40.9,43.992664,30.4,0.0,79.0,4.4,LLUVIA
223,11-08-2024,32.8,36.926356,36.0,0.0,23.0,10.0,NUBLADO
238,26-08-2024,33.2,37.358290,30.6,0.0,24.0,6.0,NUBLADO
331,27-11-2024,11.0,13.764168,3.5,0.0,32.0,NaN,NUBLADO
352,18-12-2024,13.5,15.709921,7.1,0.0,40.0,11.8,NUBLADO
358,24-12-2024,11.4,15.411886,5.3,0.3,40.0,7.9,NUBLADO



===== 2. VALORES EXTREMOS EN TEMPERATURA (>50°C o <-20°C) =====
Total encontrados: 10


,fecha,temperatura_max,temperatura_min,temperatura_media,precipitacion_mm,humedad_relativa,viento_max_kmh,condicion
39,09-02-2024,12.700000,-21.776609,5.4,0.0,28.0,11.0,NUBLADO
55,25-02-2024,59.945820,10.000000,11.1,-3.5,NaN,20.9,LLUVIA
59,29-02-2024,52.581039,9.600000,11.1,0.0,40.0,15.0,NUBLADO
62,03-03-2024,12.800000,-20.824570,7.8,0.0,56.0,22.0,DESPEJADO
68,09-03-2024,55.532906,11.900000,15.2,0.0,76.0,7.8,NUBLADO
81,22-03-2024,21.500000,-22.221888,16.0,0.0,38.0,25.9,NUBLADO
177,26-06-2024,55.204494,33.900000,41.1,NaN,73.0,0.6,NUBLADO
209,28-07-2024,50.800000,22.600000,36.8,0.0,146.0,11.5,NUBLADO
245,02-09-2024,32.800000,-20.972362,21.5,0.0,77.0,12.4,NUBLADO
326,22-11-2024,52.018418,1.400000,11.2,0.0,29.0,15.4,DESPEJADO



===== 3. PRECIPITACIONES NEGATIVAS =====
Total encontrados: 8


,fecha,temperatura_max,temperatura_min,temperatura_media,precipitacion_mm,humedad_relativa,viento_max_kmh,condicion
21,22-01-2024,9.70000,6.0,8.5,-4.8,82.0,20.9,NUBLADO
55,25-02-2024,59.94582,10.0,11.1,-3.5,NaN,20.9,LLUVIA
91,01-04-2024,25.80000,9.8,16.9,-1.7,68.0,0.9,DESPEJADO
96,06-04-2024,25.10000,12.1,18.0,-2.1,63.0,14.7,NUBLADO
152,01-06-2024,34.80000,27.2,33.9,-2.2,41.0,21.2,DESPEJADO
183,02-07-2024,41.30000,28.1,35.7,-3.9,71.0,27.8,DESPEJADO
195,14-07-2024,41.10000,36.2,36.1,-3.2,51.0,NaN,DESPEJADO
297,24-10-2024,22.00000,9.4,13.0,-1.2,70.0,13.3,DESPEJADO



===== 4. HUMEDAD FUERA DE RANGO (0-100%) =====
Total encontrados: 5


,fecha,temperatura_max,temperatura_min,temperatura_media,precipitacion_mm,humedad_relativa,viento_max_kmh,condicion
125,05-05-2024,38.1,25.9,27.9,0.0,125.0,9.7,NUBLADO
207,26-07-2024,40.9,28.7,32.7,0.8,110.0,18.3,DESPEJADO
209,28-07-2024,50.8,22.6,36.8,0.0,146.0,11.5,NUBLADO
311,07-11-2024,21.0,13.0,12.6,5.1,117.0,28.7,DESPEJADO
325,21-11-2024,11.1,8.3,13.3,0.0,119.0,20.5,NUBLADO



===== 5. REGISTROS CON VALORES NULOS (NaN) =====
Total encontrados: 24


,fecha,temperatura_max,temperatura_min,temperatura_media,precipitacion_mm,humedad_relativa,viento_max_kmh,condicion
14,15-01-2024,4.900000,-2.400000,5.6,0.0,NaN,27.9,NUBLADO
22,23-01-2024,10.700000,3.300000,2.6,0.0,NaN,31.7,NUBLADO
55,25-02-2024,59.945820,10.000000,11.1,-3.5,NaN,20.9,LLUVIA
83,24-03-2024,19.400000,7.700000,11.5,0.0,51.0,NaN,NUBLADO
85,26-03-2024,19.900000,14.300000,19.1,NaN,55.0,3.8,NUBLADO
88,29-03-2024,20.600000,10.600000,15.6,4.6,27.0,NaN,NUBLADO
101,11-04-2024,24.300000,17.800000,18.2,NaN,69.0,19.7,DESPEJADO
103,13-04-2024,23.600000,14.300000,21.7,0.0,NaN,18.0,LLUVIA
104,14-04-2024,25.800000,15.500000,25.9,NaN,22.0,22.7,DESPEJADO
128,08-05-2024,32.500000,26.800000,27.8,0.0,NaN,17.9,NUBLADO


SOLUCIONAR MIN > MAX

In [87]:

# 2. Identificar la máscara donde la mínima es mayor que la máxima
mask_swap = meteo_limpio['temperatura_min'] > meteo_limpio['temperatura_max']

# 3. Intercambiar los valores de 'temperatura_max' y 'temperatura_min' en esas filas
meteo_limpio.loc[mask_swap, ['temperatura_max', 'temperatura_min']] = (
    meteo_limpio.loc[mask_swap, ['temperatura_min', 'temperatura_max']].values
)

# 4. Comprobación: verificar que ya no existen casos donde min > max
casos_restantes = len(meteo_limpio[meteo_limpio['temperatura_min'] > meteo_limpio['temperatura_max']])
print(f"Casos donde Temp. Min > Temp. Max después de la inversión: {casos_restantes}")

Casos donde Temp. Min > Temp. Max después de la inversión: 0


SOLUCIÓN TEMP EXTREMAS

Hemos decicido que descartaremos aquellas temperaturas > 50 o < -15, ya que las consideramos imposibles. Y revisaremos aquellas que se encuentren en el rengo de 45 a 50 grados. De - 15 a -10 también es un rango a revisar, sin embargo no tenemos ningún dato en él. Mantendremos aquellas que esten entre -10 y 45 grados. 

In [88]:
# Descartar < -15 y > 50
mask_min_erronea = meteo_limpio['temperatura_min'] < -15
mask_max_erronea = meteo_limpio['temperatura_max'] > 50

# Convertir a NaN las temperaturas imposibles 
meteo_limpio.loc[mask_min_erronea, 'temperatura_min'] = np.nan
meteo_limpio.loc[mask_max_erronea, 'temperatura_max'] = np.nan

# Casos en zona de revisión (-15 a -10 °C y 45 a 50 °C)
revisar_min = meteo_limpio[(meteo_limpio['temperatura_min'] >= -15) & (meteo_limpio['temperatura_min'] <= -10)]
revisar_max = meteo_limpio[(meteo_limpio['temperatura_max'] >= 45) & (meteo_limpio['temperatura_max'] <= 50)]

print(f" Temperaturas mínimas descartadas (< -15 °C): {mask_min_erronea.sum()}")
print(f" Temperaturas máximas descartadas (> 50 °C): {mask_max_erronea.sum()}")
print(f" Registros a revisar en mínimas (-15 °C a -10 °C): {len(revisar_min)}")
print(f" Registros a revisar en máximas (45 °C a 50 °C): {len(revisar_max)}")

 Temperaturas mínimas descartadas (< -15 °C): 4
 Temperaturas máximas descartadas (> 50 °C): 6
 Registros a revisar en mínimas (-15 °C a -10 °C): 0
 Registros a revisar en máximas (45 °C a 50 °C): 2


In [89]:
# Mostrar detalle de registros a revisar

if not revisar_max.empty:
    print("\nREGISTROS A REVISAR (Máximas entre 45 °C y 50 °C):")
    display(revisar_max[['fecha', 'temperatura_min', 'temperatura_max', 'temperatura_media']])
else:
    print("\nNo hay registros de máximas en el rango 45 °C a 50 °C.")


REGISTROS A REVISAR (Máximas entre 45 °C y 50 °C):


,fecha,temperatura_min,temperatura_max,temperatura_media
179,28-06-2024,27.3,47.9,35.4
220,08-08-2024,26.7,45.0,29.3


Decidimos descartar las dos temperaturas maximas.
--- Justificación --> 

* **Récord superado:** Ambos valores exceden los máximos históricos de ls distritos con los que trabajamos (42.7 en Barajas).
* **Anomalía estacional:** Alcanzar 47.9 en junio (28-06) o 45.0 en agosto (08-08) resulta atípico para el clima local.
* **Inconsistencia matemática:** El día 08-08 se registra una media de 29.3, incompatible con una máxima de 45.0 y mínima de 26.7 (la máxima real debió rondar los 31.9).
Por loq eu se reemplazan ambos valores por NaN.

In [90]:
# Convertir las máximas de estos dos días específicos a NaN 
meteo_limpio.loc[(meteo_limpio['fecha'] == '28-06-2024') & (meteo_limpio['temperatura_max'] == 47.9), 'temperatura_max'] = np.nan
meteo_limpio.loc[(meteo_limpio['fecha'] == '08-08-2024') & (meteo_limpio['temperatura_max'] == 45.0), 'temperatura_max'] = np.nan

print("Valores atípicos de la zona de revisión ajustados a NaN.")

Valores atípicos de la zona de revisión ajustados a NaN.


SOLUCIÓN PRECIPITACIONES NEGATIVAS Y HUMEDAD FUERA DE RANGO 

In [91]:
# Precipitaciones negativas (< 0 mm) -> NaN
mask_precip_negativa = meteo_limpio['precipitacion_mm'] < 0
meteo_limpio.loc[mask_precip_negativa, 'precipitacion_mm'] = np.nan

# Humedad relativa fuera de rango (< 0% o > 100%) -> NaN
mask_humedad_fuera_rango = (meteo_limpio['humedad_relativa'] < 0) | (meteo_limpio['humedad_relativa'] > 100)
meteo_limpio.loc[mask_humedad_fuera_rango, 'humedad_relativa'] = np.nan

# 3. Comprobación de registros descartados
print(f"Precipitaciones negativas convertidas a NaN: {mask_precip_negativa.sum()}")
print(f"Humedades fuera de rango convertidas a NaN: {mask_humedad_fuera_rango.sum()}")
print("=" * 60)

# Verificación de que no quedan valores fuera de límites
print(f"Precipitaciones mínimas actuales: {meteo_limpio['precipitacion_mm'].min()} mm")
print(f"Humedad mínima actual: {meteo_limpio['humedad_relativa'].min()} %")
print(f"Humedad máxima actual: {meteo_limpio['humedad_relativa'].max()} %")

Precipitaciones negativas convertidas a NaN: 8
Humedades fuera de rango convertidas a NaN: 5
Precipitaciones mínimas actuales: 0.0 mm
Humedad mínima actual: 20.0 %
Humedad máxima actual: 84.0 %


COMPROBACIÓN COHERENCIA TEMP MEDIA, CONDICIÓN, PRECIPITACIÓN, ETC

In [92]:

# Convertir la columna fecha a Datetime temporalmente para extraer el mes
df_temp_check = meteo_limpio.copy()
df_temp_check['fecha_dt'] = pd.to_datetime(df_temp_check['fecha'], format='%d-%m-%Y')
df_temp_check['mes'] = df_temp_check['fecha_dt'].dt.month

# 1. COMPROBACIÓN MATEMÁTICA: Media teórica vs Media registrada
df_temp_check['media_teorica'] = (df_temp_check['temperatura_max'] + df_temp_check['temperatura_min']) / 2
df_temp_check['dif_media'] = (df_temp_check['temperatura_media'] - df_temp_check['media_teorica']).abs()

dias_desviados = df_temp_check[df_temp_check['dif_media'] > 3.0] # Diferencia mayor a 3°C
print(f"1. Días donde T_media difiere >3°C de (T_max + T_min)/2: {len(dias_desviados)}")
if not dias_desviados.empty:
    display(dias_desviados[['fecha', 'temperatura_min', 'temperatura_max', 'temperatura_media', 'media_teorica']])

# 2. COMPROBACIÓN ESTACIONAL: Rango de Temperatura Media por Mes
print("\n2. Rango de Temperatura Media (°C) por Mes:")
resumen_mensual = df_temp_check.groupby('mes')['temperatura_media'].agg(['min', 'mean', 'max'])
display(resumen_mensual)

# 3. COMPROBACIÓN DE SALTOS TÉRMICOS BRUSCOS (Variación diaria > 15°C)
df_temp_check['salto_diario'] = df_temp_check['temperatura_media'].diff().abs()
saltos_extremos = df_temp_check[df_temp_check['salto_diario'] > 15]
print(f"\n3. Días con saltos bruscos de T_media (>15°C en 24h): {len(saltos_extremos)}")
if not saltos_extremos.empty:
    display(saltos_extremos[['fecha', 'temperatura_media', 'salto_diario']])

# 4. COMPROBACIÓN DE COHERENCIA ENTRE LLUVIA Y CONDICION
lluvia_despejado = df_temp_check[(df_temp_check['precipitacion_mm'] > 0) & (df_temp_check['condicion'] == 'DESPEJADO')]
print(f"\n4. Días con lluvia (>0mm) marcados como 'DESPEJADO': {len(lluvia_despejado)}")
if not lluvia_despejado.empty:
    display(lluvia_despejado[['fecha', 'precipitacion_mm', 'condicion']])

1. Días donde T_media difiere >3°C de (T_max + T_min)/2: 138


,fecha,temperatura_min,temperatura_max,temperatura_media,media_teorica
1,02-01-2024,0.1,9.700000,-0.0,4.900000
8,09-01-2024,6.5,8.600000,-0.5,7.550000
9,10-01-2024,-2.3,11.600000,1.2,4.650000
14,15-01-2024,-2.4,4.900000,5.6,1.250000
17,18-01-2024,-1.4,11.100000,1.0,4.850000
20,21-01-2024,1.4,14.700000,2.4,8.050000
22,23-01-2024,3.3,10.700000,2.6,7.000000
23,24-01-2024,-1.2,6.200000,13.4,2.500000
24,25-01-2024,-2.1,8.900000,7.1,3.400000
30,31-01-2024,-4.1,9.300000,13.8,2.600000



2. Rango de Temperatura Media (°C) por Mes:


,min,mean,max
mes,,,
1,-0.5,5.274194,13.8
2,3.0,8.651724,12.6
3,7.0,14.545161,24.6
4,15.7,21.533333,31.4
5,22.9,29.606452,38.8
6,30.3,35.183333,41.1
7,28.1,34.735484,40.5
8,23.6,31.932258,37.3
9,19.5,25.586667,31.3



3. Días con saltos bruscos de T_media (>15°C en 24h): 0

4. Días con lluvia (>0mm) marcados como 'DESPEJADO': 41


,fecha,precipitacion_mm,condicion
1,02-01-2024,0.1,DESPEJADO
3,04-01-2024,0.6,DESPEJADO
12,13-01-2024,10.6,DESPEJADO
13,14-01-2024,3.6,DESPEJADO
19,20-01-2024,2.1,DESPEJADO
23,24-01-2024,2.5,DESPEJADO
26,27-01-2024,2.3,DESPEJADO
27,28-01-2024,3.8,DESPEJADO
42,12-02-2024,0.1,DESPEJADO
58,28-02-2024,0.3,DESPEJADO


---

In [ ]:
# Guardar SOLO el dataset limpio
meteo_limpio.to_csv(
    'meteo_limpio.csv',
    index=False
)

print("Guardado correctamente como 'meteo_limpio_con_nulos.csv'")
print(f"Dimensiones finales: {meteo_limpio.shape}")

Guardado correctamente como 'meteo_limpio.csv'
Dimensiones finales: (366, 10)
